# Constraint-reconstruction benchmark — GPU sweep

Runs the full benchmark grid (the three curves from `benchmarks/README.md`) on a
Colab GPU and downloads the `results/*.jsonl` at the end.

**Setup:** Runtime → Change runtime type → **GPU** (T4 is fine), then Run all.

Results append to `results/<dataset>.jsonl` exactly like `python -m benchmarks.run`,
so a re-run after a disconnect just adds rows — the plotting cell dedupes on
`(engine, n_pair, noise, conflict, seed)` keeping the latest.

In [ ]:
# --- Setup: clone repo, install the two deps Colab lacks, keep Colab's own jax ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/amdson/calibrated_response.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)

# optax/jaxopt are the only benchmark deps Colab doesn't ship; installing the
# package itself would drag in pinned jax/jaxlib and clobber the GPU build.
%pip -q install optax jaxopt

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"

In [ ]:
# --- Sweep configuration ---
# QUICK=True: shrunken grid to validate the whole notebook end-to-end (~minutes).
# QUICK=False: the real grid behind the three README curves.
QUICK = True

DATASETS = ["synthetic_chain", "adult", "bank_marketing", "california", "nursery"]
ENGINES  = ["independent", "copula", "gaussian", "tn", "tree"]  # add "flow" if time allows
SEEDS    = [0] if QUICK else [0, 1, 2]
N_BINS   = 8
MAX_ROWS = 4000 if QUICK else None

# The three curves: each entry is one call to benchmarks.run.main per dataset.
if QUICK:
    SWEEPS = [
        dict(tag="nll_vs_npair",    n_pair=[20, 40],       noise=[0.0],       conflict=[0.0]),
        dict(tag="nll_vs_noise",    n_pair=[40],           noise=[0.0, 0.6],  conflict=[0.0]),
        dict(tag="nll_vs_conflict", n_pair=[40],           noise=[0.3],       conflict=[0.0, 0.2]),
    ]
else:
    SWEEPS = [
        dict(tag="nll_vs_npair",    n_pair=[10, 20, 40, 80], noise=[0.0],                        conflict=[0.0]),
        dict(tag="nll_vs_noise",    n_pair=[40],             noise=[0.0, 0.15, 0.3, 0.6, 1.0],   conflict=[0.0]),
        dict(tag="nll_vs_conflict", n_pair=[40],             noise=[0.3],                        conflict=[0.0, 0.1, 0.2]),
    ]

n_cells = sum(len(s["n_pair"]) * len(s["noise"]) * len(s["conflict"]) for s in SWEEPS)
print(f"{n_cells} grid points x {len(SEEDS)} seeds x {len(ENGINES)} engines x "
      f"{len(DATASETS)} datasets = {n_cells * len(SEEDS) * len(ENGINES) * len(DATASETS)} fits")

In [ ]:
# --- Run the sweeps (resumable: rows append to results/<dataset>.jsonl) ---
import time
from benchmarks.run import main as run_main

t_start = time.time()
for dataset in DATASETS:
    for sweep in SWEEPS:
        print(f"\n=== {dataset} / {sweep['tag']} ({time.time() - t_start:.0f}s elapsed) ===")
        argv = ["--dataset", dataset,
                "--engines", *ENGINES,
                "--n-pair", *map(str, sweep["n_pair"]),
                "--noise", *map(str, sweep["noise"]),
                "--conflict", *map(str, sweep["conflict"]),
                "--seeds", *map(str, SEEDS),
                "--n-bins-cont", str(N_BINS)]
        if MAX_ROWS:
            argv += ["--max-rows", str(MAX_ROWS)]
        run_main(argv)
print(f"\nDone in {(time.time() - t_start) / 60:.1f} min")

In [ ]:
# --- Plot the three curves per dataset ---
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

KEY = ["engine", "n_pair", "noise", "conflict", "seed"]

def load_results(dataset):
    rows = [json.loads(l) for l in Path(f"benchmarks/results/{dataset}.jsonl").read_text().splitlines()]
    return pd.DataFrame(rows).drop_duplicates(KEY, keep="last")

def curve(ax, df, x, y="heldout_nll", **fixed):
    sub = df
    for k, v in fixed.items():
        sub = sub[sub[k] == v]
    for eng, g in sub.groupby("engine"):
        m = g.groupby(x)[y].agg(["mean", "sem"])
        ax.errorbar(m.index, m["mean"], yerr=m["sem"], marker="o", label=eng, capsize=3)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.grid(alpha=0.3)

for dataset in DATASETS:
    if not Path(f"benchmarks/results/{dataset}.jsonl").exists():
        continue
    df = load_results(dataset)
    for y in ["heldout_nll", "pair_tv_unseen"]:
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
        curve(axes[0], df, "n_pair", y, noise=0.0, conflict=0.0)
        curve(axes[1], df, "noise", y, n_pair=40, conflict=0.0)
        curve(axes[2], df, "conflict", y, n_pair=40, noise=0.3)
        axes[0].legend(fontsize=8)
        fig.suptitle(f"{dataset} — {y}")
        plt.tight_layout(); plt.show()

In [ ]:
# --- Download results (unzip into benchmarks/results/ locally to merge) ---
import shutil
shutil.make_archive("/content/benchmark_results", "zip", "benchmarks/results")
try:
    from google.colab import files
    files.download("/content/benchmark_results.zip")
except ImportError:
    print("not on Colab — results in benchmarks/results/")